In [1]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from PIL import Image
import numpy as np

In [26]:
from pathlib import Path

from PIL import Image

import numpy as np

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# 1. Dataset path
dataset_path = Path(
    r"D:\good_projects\visionguard\mvtec_anomaly_detection"
)
# 2. Path to good training images
train_good_path = dataset_path / "bottle" / "train" / "good"

# 3. Get an image path
image_path = next(train_good_path.iterdir())

print("Image path:", image_path)

# 4. Open the image
image = Image.open(image_path).convert("RGB")

print("Original size:", image.size)

# 5. Resize the image
image = image.resize((224, 224))

print("New size:", image.size)

# 6. Convert PIL image → NumPy array

image_array = np.array(image)

print("After NumPy conversion:", image_array.shape)

# 7. Add batch dimension
image_array = np.expand_dims(image_array, axis=0)

print("After adding batch dimension:", image_array.shape) #The 1 means:We are passing one image in this batch.


image_array = preprocess_input(image_array)


model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


features = model.predict(image_array, verbose=0)

print("Feature shape:", features.shape)

Image path: D:\good_projects\visionguard\mvtec_anomaly_detection\bottle\train\good\000.png
Original size: (900, 900)
New size: (224, 224)
After NumPy conversion: (224, 224, 3)
After adding batch dimension: (1, 224, 224, 3)
Feature shape: (1, 7, 7, 512)


In [27]:
model.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
embedding = features.flatten()

print("Embedding shape:", embedding.shape)

Embedding shape: (25088,)


In [29]:
def extract_embedding(image_path, model):
    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224))

    image_array = np.array(image)
    image_array = np.expand_dims(image_array, axis=0)
    image_array = preprocess_input(image_array)

    features = model.predict(image_array, verbose=0)

    embedding = features.flatten()

    return embedding

In [31]:
image_path = next(train_good_path.iterdir())

embedding = extract_embedding(image_path, model)

print(embedding.shape)

print(embedding[:10])

(25088,)
[-0. -0. -0. -0. -0. -0. -0. -0. -0. -0.]


In [32]:
normal_embeddings = []

for image_path in train_good_path.iterdir():

    if image_path.is_file():

        embedding = extract_embedding(image_path, model)

        normal_embeddings.append(embedding)

In [33]:
normal_embeddings = np.array(normal_embeddings)
print(normal_embeddings.shape)

(209, 25088)


In [34]:
test_path = dataset_path / "bottle" / "test" / "good"

test_image = next(test_path.iterdir())

test_embedding = extract_embedding(test_image, model)

In [36]:
distances = np.linalg.norm(
    normal_embeddings - test_embedding,
    axis=1
)
print(distances.shape)

(209,)


In [37]:
anomaly_score = distances.min()

print("Anomaly score:", anomaly_score)

Anomaly score: 282.36252


In [38]:
test_root = dataset_path / "bottle" / "test"

test_categories = [
    item for item in test_root.iterdir()
    if item.is_dir()
]

print(test_categories)

[WindowsPath('D:/good_projects/visionguard/mvtec_anomaly_detection/bottle/test/broken_large'), WindowsPath('D:/good_projects/visionguard/mvtec_anomaly_detection/bottle/test/broken_small'), WindowsPath('D:/good_projects/visionguard/mvtec_anomaly_detection/bottle/test/contamination'), WindowsPath('D:/good_projects/visionguard/mvtec_anomaly_detection/bottle/test/good')]


In [39]:
broken_small_path = test_root / "broken_small"

broken_image = next(broken_small_path.iterdir())

broken_embedding = extract_embedding(
    broken_image,
    model
)

In [40]:
broken_distances = np.linalg.norm(
    normal_embeddings - broken_embedding,
    axis=1
)

broken_score = broken_distances.min()

print("Broken image anomaly score:", broken_score)

Broken image anomaly score: 386.90942


In [41]:
def calculate_anomaly_score(
    embedding,
    normal_embeddings
):
    distances = np.linalg.norm(
        normal_embeddings - embedding,
        axis=1
    )

    return distances.min()

In [42]:
normal_score = calculate_anomaly_score(
    test_embedding,
    normal_embeddings
)

broken_score = calculate_anomaly_score(
    broken_embedding,
    normal_embeddings
)

print("Normal:", normal_score)
print("Broken:", broken_score)

Normal: 282.36252
Broken: 386.90942
